In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 12. XGBoost Training - Pressor (승압제 예측)

## 목적
Pressor 모델 학습 및 평가 (3개 시간대)

## 예측 타겟
| 모델 | 타겟 | 예측 시점 |
|------|------|----------|
| Pressor 6h | pressor_next_6h | 6시간 내 승압제 |
| Pressor 12h | pressor_next_12h | 12시간 내 승압제 |
| Pressor 24h | pressor_next_24h | 24시간 내 승압제 |

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import sys
import xgboost as xgb
from datetime import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt

# metrics.py 경로 추가
sys.path.append('/content/drive/MyDrive/project/mini_20260113')
from metrics import (
    plot_roc_curves, plot_pr_curves, plot_combined_roc,
    print_confusion_matrices, plot_feature_importance
)

print("=== 12. XGBoost Training - Pressor ===")
print(f"XGBoost version: {xgb.__version__}")

## 설정

In [ ]:
# 경로 설정
DATA_DIR = '/content/drive/MyDrive/project/mini_20260113/data/pressor'
OUTPUT_DIR = '/content/drive/MyDrive/project/mini_20260113/models/pressor'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGETS = {
    'pressor_6h': 'pressor_next_6h',
    'pressor_12h': 'pressor_next_12h',
    'pressor_24h': 'pressor_next_24h'
}

BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'aucpr'],
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS = 500

print(f"데이터 경로: {DATA_DIR}")
print(f"모델 저장 경로: {OUTPUT_DIR}")

## Step 1: 데이터 로드

In [ ]:
print("Step 1: 데이터 로드")

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"✓ Train: {len(train):,} rows")
print(f"✓ Val: {len(val):,} rows")
print(f"✓ Test: {len(test):,} rows")

with open(os.path.join(DATA_DIR, 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)
print(f"✓ 피처: {len(feature_cols)}개")

with open(os.path.join(DATA_DIR, 'scale_pos_weights.json'), 'r') as f:
    scale_pos_weights = json.load(f)
print(f"✓ 클래스 가중치 로드 완료")

In [ ]:
X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

print(f"피처 shape:")
print(f" X_train: {X_train.shape}")
print(f" X_val: {X_val.shape}")
print(f" X_test: {X_test.shape}")

print(f"\n=== 타겟 레이블 분포 (Train) ===")
for name, col in TARGETS.items():
    pos_rate = train[col].mean() * 100
    pos_count = train[col].sum()
    weight = scale_pos_weights.get(col, 1)
    print(f" {name}: {pos_count:,} ({pos_rate:.2f}%) | scale_pos_weight={weight:.1f}")

## Step 2: 모델 학습

In [ ]:
print("Step 2: 모델 학습")

models = {}
training_history = {}

for name, target_col in TARGETS.items():
    print(f"\n{'='*50}")
    print(f"[{name.upper()}] Training: {target_col}")
    print(f"{'='*50}")

    y_train = train[target_col]
    y_val = val[target_col]

    weight = scale_pos_weights.get(target_col, 1)
    print(f" scale_pos_weight: {weight:.1f}")

    params = BASE_PARAMS.copy()
    params['scale_pos_weight'] = weight

    model = xgb.XGBClassifier(
        **params,
        n_estimators=N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )

    models[name] = model
    best_iteration = model.best_iteration
    results = model.evals_result()
    training_history[name] = results

    print(f" ✓ Best iteration: {best_iteration}")
    print(f" ✓ Train AUC: {results['validation_0']['auc'][best_iteration]:.4f}")
    print(f" ✓ Val AUC: {results['validation_1']['auc'][best_iteration]:.4f}")

print(f"\n{'='*50}")
print("✓ 모든 모델 학습 완료!")
print(f"{'='*50}")

## Step 3: 모델 평가 (Test Set)

In [ ]:
# 최적 Threshold 설정
THRESHOLDS = {
    'pressor_6h': 0.20,
    'pressor_12h': 0.20,
    'pressor_24h': 0.20
}

In [ ]:
print("Step 3: 모델 평가 (Test Set)")

results = {}
y_true_dict = {}
y_prob_dict = {}

for name, col in TARGETS.items():
    y_true = test[col]
    y_prob = models[name].predict_proba(X_test)[:, 1]
    thresh = THRESHOLDS[name]
    y_pred = (y_prob >= thresh).astype(int)

    y_true_dict[name] = y_true
    y_prob_dict[name] = y_prob

    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results[name] = {
        'threshold': thresh,
        'auroc': auroc,
        'auprc': auprc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'tn': int(tn)
    }

print("=" * 80)
print(" XGBoost - Pressor Performance")
print("=" * 80)
print(f"{'Model':<14} {'Thresh':>6} {'AUROC':>8} {'AUPRC':>8} {'F1':>8} {'Prec':>8} {'Recall':>8} {'Spec':>8}")
print("-" * 80)
for name, r in results.items():
    print(f"{name:<14} {r['threshold']:>6.2f} {r['auroc']:>8.4f} {r['auprc']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f} {r['specificity']:>8.4f}")
print("-" * 80)

In [ ]:
# TP/FP/FN 출력
print("[Detection Summary]")
print(f"{'Model':<14} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'Caught%':>10}")
print("-" * 62)
for name, r in results.items():
    total_pos = r['tp'] + r['fn']
    caught_pct = r['tp'] / total_pos * 100 if total_pos > 0 else 0
    print(f"{name:<14} {r['tp']:>8} {r['fp']:>8} {r['fn']:>8} {r['tn']:>8} {caught_pct:>9.1f}%")

In [ ]:
# Confusion Matrix 시각화
print_confusion_matrices(results)

## Step 4: 시각화

In [ ]:
# ROC Curves (3개 모델)
plot_roc_curves(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(OUTPUT_DIR, 'roc_curves.png')
)

In [ ]:
# PR Curves (3개 모델)
plot_pr_curves(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(OUTPUT_DIR, 'pr_curves.png')
)

In [ ]:
# Combined ROC (단일 그래프)
plot_combined_roc(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(OUTPUT_DIR, 'roc_combined.png')
)

## Step 5: Feature Importance

In [ ]:
# Pressor 12h 모델 피처 중요도 (대표)
importance_dict = plot_feature_importance(
    models['pressor_12h'],
    feature_cols,
    top_n=13,
    save_path=os.path.join(OUTPUT_DIR, 'feature_importance.png'),
    title='Feature Importance (Pressor 12h Model)'
)

In [ ]:
# 3개 모델 피처 중요도 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = {
    'pressor_6h': 'Pressor 6h',
    'pressor_12h': 'Pressor 12h',
    'pressor_24h': 'Pressor 24h'
}

for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx]

    importance = model.feature_importances_
    indices = np.argsort(importance)[::-1][:13]

    y_pos = np.arange(len(indices))
    ax.barh(y_pos, [importance[i] for i in indices[::-1]], color='coral')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_cols[i] for i in indices[::-1]], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title(titles[name], fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance_all.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Feature importance (all models) saved")

## Step 6: 모델 저장

In [ ]:
print("Step 6: 모델 저장")

for name, model in models.items():
    model_path = os.path.join(OUTPUT_DIR, f'{name}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f" ✓ {name}: {model_path}")

results_path = os.path.join(OUTPUT_DIR, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Results saved: {results_path}")

In [ ]:
# 학습 설정 저장 (thresholds 추가)
config = {
    'model_type': 'XGBoost',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'targets': TARGETS,
    'thresholds': THRESHOLDS,
    'params': BASE_PARAMS,
    'n_estimators': N_ESTIMATORS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'best_iterations': {name: int(model.best_iteration) for name, model in models.items()},
    'results': results
}

config_path = os.path.join(OUTPUT_DIR, 'training_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"✓ Training config saved: {config_path}")

## Step 7: 최종 요약

In [ ]:
print("="*70)
print(" 12. XGBoost Training - Pressor 완료")
print("="*70)

print(f"\n📊 성능 요약 (Test Set):")
print(f"{'Model':<14} {'AUROC':>10} {'AUPRC':>10}")
print("-" * 37)
for name, metrics in results.items():
    print(f"{name:<14} {metrics['auroc']:>10.4f} {metrics['auprc']:>10.4f}")

avg_auroc = np.mean([m['auroc'] for m in results.values()])
avg_auprc = np.mean([m['auprc'] for m in results.values()])
print("-" * 37)
print(f"{'Average':<14} {avg_auroc:>10.4f} {avg_auprc:>10.4f}")

print(f"\n📁 저장된 파일:")
print(f" - 모델: {OUTPUT_DIR}/*.pkl")
print(f" - 결과: {OUTPUT_DIR}/results.json")
print(f" - 설정: {OUTPUT_DIR}/training_config.json")
print(f" - 시각화: {OUTPUT_DIR}/*.png")